# Replication: Baron (1982) — Regulating a Monopolist with Unknown Costs (Python)

## Goal of this notebook
We replicate the optimal regulation policy characterized in Baron (1982) by:
1) computing the virtual/adjusted cost function,
2) "ironing" it into a monotone function,
3) constructing the optimal policy (price, quantity, shutdown, subsidy),
4) simulating outcomes over parameters and distributions.

## What we replicate from the paper
- Cost: $C(q,θ) = (c_0 + c_1 θ) q + (k_0 + k_1 θ) \text{ for } q>0$, and $C(0,θ)=0$. 
- Demand: $P(q) = V'(q)$. 
- Virtual cost: $z_a(θ) = θ + (1-a) F(θ)/f(θ)$. 
- Ironing construction (CDF-space convexification): eqs (19)-(23). 
- Policy rules: eqs (25)-(28). 

## Notebook structure
1. Imports and numerical helpers
2. Model primitives (demand, value, costs)
3. Type distributions F,f and numerical inverse CDF
4. Virtual cost z_a(θ) and ironing to get z_fa(θ)
5. Optimal policy mapping (p,q,r,π,s)
6. Replication check using the paper’s worked example
7. Simulation loops → pandas DataFrame
8. Figures + economic interpretation helpers
9. Extension: how distribution shape affects ironing ("bunching") and outcomes


In [ ]:
!pip install numpy pandas matplotlib scipy

In [1]:
# Core numerical stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

In [2]:


def cumtrapz(y, x):
    """
    LaTeX (raw):
        \\int y(x) dx  \\approx  \\sum_{i=1}^n \\frac{(y_i + y_{i-1})}{2} (x_i - x_{i-1})
    """
    # cumulative trapezoid integral with same length as x (starts at 0)
    y = np.asarray(y)
    x = np.asarray(x)
    out = np.zeros_like(x, dtype=float)
    dx = np.diff(x)
    out[1:] = np.cumsum(0.5 * (y[1:] + y[:-1]) * dx)
    return out


def safe_div(num, den, eps=1e-12):
    """
    LaTeX (raw):
        \\frac{num}{den} \\approx \\frac{num}{\\max(den, \\varepsilon)}
    """
    return np.asarray(num) / np.maximum(np.asarray(den), eps)


def monotone_decreasing(x, y):
    """
    Quick diagnostic: checks if y(x) is weakly decreasing in x.
    """
    idx = np.argsort(x)
    y_sorted = np.asarray(y)[idx]
    return np.all(np.diff(y_sorted) <= 1e-10)


# 1. Model primitives

## Demand / value
The paper uses a general concave value function $V(q)$, with inverse demand $P(q)=V'(q)$.

In code, we implement demand as:
- `P(q)` = inverse demand (price as a function of quantity)
- `V(q)` = integral of P(q)

We will include:
1) Linear inverse demand: $P(q)=A - B q$
2) Isoelastic inverse demand: $P(q)=A q^{-1/ε} (ε>1)$

## Costs
The firm cost is bilinear in (q,θ):
$C(q,θ) = (c_0 + c_1 θ) q + (k_0 + k_1 θ)  \text{ for } q>0;  C(0,θ)=0.$ 

We'll code it as `cost(q, theta, params)` and use it to compute profits and subsidies.


In [3]:
def P_linear(q, A, B):
    r"""
    LaTeX (raw):
        P(q) = A - B q
    """
    q = np.asarray(q)
    return A - B * q


def V_linear(q, A, B):
    r"""
    LaTeX (raw):
        V(q) = \int_0^q (A - B t)\,dt = A q - \frac{B}{2} q^2
    """
    q = np.asarray(q)
    return A * q - 0.5 * B * q**2


def P_isoelastic(q, A, eps):
    r"""
    LaTeX (raw):
        P(q) = A q^{-1/\varepsilon}, \quad \varepsilon>1
    """
    q = np.asarray(q)
    q = np.maximum(q, 1e-12)  # avoid division by zero
    return A * q ** (-1.0 / eps)


def V_isoelastic(q, A, eps):
    r"""
    LaTeX (raw):
        V(q) = \int_0^q A t^{-1/\varepsilon} dt
             = A \frac{\varepsilon}{\varepsilon-1} q^{(\varepsilon-1)/\varepsilon} \quad (\varepsilon \ne 1)
    """
    q = np.asarray(q)
    q = np.maximum(q, 1e-12)
    return A * (eps / (eps - 1.0)) * q ** ((eps - 1.0) / eps)


def cost(q, theta, c0, c1, k0, k1):
    r"""
    LaTeX (raw):
        C(q,\theta) =
        \begin{cases}
        (c_0 + c_1 \theta) q + (k_0 + k_1 \theta) & q>0 \\
        0 & q=0
        \end{cases}
    """
    q = np.asarray(q)
    theta = np.asarray(theta)
    C = (c0 + c1 * theta) * q + (k0 + k1 * theta)
    return np.where(q > 0, C, 0.0)


# 2. Quantity from a regulated price

Policy equation (26) imposes that the chosen quantity satisfies:
$P(q(θ)) = p(θ)$. 

Computationally, we can obtain q in two equivalent ways:
1) Root/fixed-point: solve $P(q) - p = 0$
2) Optimization: q solves $\max_{q>=0} [V(q) - p q]$, whose FOC is $V'(q)=P(q)=p$.

We'll implement the optimization method (good for the assignment requirement),
and provide a fallback grid-search if SciPy isn't available.
